<center><a href="https://www.nvidia.com/dli"> <img src="images/DLI_Header.png" alt="Header" style="width: 400px;"/> </a></center>




# 7. 評量(Assessment)

恭喜您完成今天的課程！希望您在學習過程中獲得了一些寶貴的技能，並且樂在其中。現在是時候測試這些技能了。在這次評量中，您將訓練一個能夠識別新鮮和腐爛水果的新模型。您需要讓模型在驗證(Validation)資料集上達到`92%`的準確度(accuracy)才能通過評量，不過我們鼓勵您盡可能做得更好。您將需要運用在先前練習中學到的技能。具體來說，我們建議使用遷移學習(transfer learning)、資料增強(data augmentation)和微調(fine tuning)的某種組合。一旦您訓練的模型在驗證資料集上至少達到92%的準確度，請儲存您的模型，然後評估其準確度。讓我們開始吧！

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as transforms
import torchvision.io as tv_io

import glob
from PIL import Image

import utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

## 7.1 資料集

在這個練習中，您將訓練一個模型來識別新鮮和腐爛的水果。資料集來自[Kaggle](https://www.kaggle.com/sriramr/fruits-fresh-and-rotten-for-classification)，這是一個很好的平台，如果您在本課程後有興趣開始一個專案。資料集結構位於`data/fruits`資料夾中。共有6種水果類別：新鮮蘋果(fresh apples)、新鮮柳橙(fresh oranges)、新鮮香蕉(fresh bananas)、腐爛蘋果(rotten apples)、腐爛柳橙(rotten oranges)和腐爛香蕉(rotten bananas)。這意味著您的模型將需要一個包含6個神經元的輸出層才能成功進行分類。由於我們有超過兩個類別，您還需要使用`categorical_crossentropy`來編譯模型。

<img src="./images/fruits.png" style="width: 600px;">


<img src="./images/fruits.png" style="width: 600px;">

## 7.2 載入ImageNet基礎模型



我們鼓勵您從預訓練的ImageNet模型開始。使用正確的權重載入模型。由於這些圖片是彩色的，所以會有紅、綠、藍三個通道(channel)。我們已經為您填寫了輸入形狀。如果您需要設置預訓練模型的參考，請查看[notebook 05b](05b_presidential_doggy_door.ipynb)，我們在其中實做了遷移學習。

In [ ]:
from torchvision.models import vgg16
from torchvision.models import VGG16_Weights

weights = VGG16_Weights.FIXME
vgg_model = vgg16(weights=weights)


## 7.3 凍結基礎模型


接下來，我們建議凍結基礎模型，如[notebook 05b](05b_presidential_doggy_door.ipynb)中所做的那樣。這樣做是為了確保ImageNet資料集的所有學習在初始訓練中不會被破壞。

In [ ]:
# Freeze base model
vgg_model.requires_grad_(FIXME)
next(iter(vgg_model.parameters())).requires_grad

## 7.4 向模型添加層


現在是時候向預訓練模型添加層了。[Notebook 05b](05b_presidential_doggy_door.ipynb)可以作為指南。請特別注意最後的密集層(dense layer)，並確保它有正確數量的神經元來分類不同類型的水果。

模型的後面層會跟模型訓練的資料更加有關連。由於我們想要從VGG獲得更泛化的學習能力，我們可以選擇其中的部分，如下所示：

In [ ]:
vgg_model.classifier[0:3]


一旦我們從VGG16中獲取了我們想要的部分，我們就可以添加自己的改動。無論我們添加什麼額外的模組，我們仍然需要在每個輸出最後產出一個數值。

In [ ]:
N_CLASSES = FIXME

my_model = nn.Sequential(
    vgg_model.features,
    vgg_model.avgpool,
    nn.Flatten(),
    vgg_model.classifier[0:3],
    nn.Linear(4096, 500),
    nn.ReLU(),
    nn.Linear(500, N_CLASSES)
)
my_model

## 7.5 編譯模型

現在是時候設定損失函數(loss)和指標(metric)來編譯模型了。我們有6個類別，所以應該使用哪個損失函式(loss function)？

In [ ]:
loss_function = nn.FIXME()
optimizer = Adam(my_model.parameters())
my_model = torch.compile(my_model.to(device))

## 7.6 資料轉換器


為了預處理我們的輸入圖像，我們將使用VGG16權重中包含的轉換器。

In [ ]:
pre_trans = weights.transforms()


嘗試隨機增強資料以改進資料集。請隨時查看[notebook 04a](04a_asl_augmentation.ipynb)和[notebook 05b](05b_presidential_doggy_door.ipynb)中的資料增強範例。還有[TorchVision Transforms class](https://pytorch.org/vision/stable/transforms.html)的文件。

**提示**：記住不要讓資料增強處理過於極端。

In [ ]:
IMG_WIDTH, IMG_HEIGHT = (224, 224)

random_trans = transforms.Compose([
    FIXME
])


## 7.7 載入資料集


現在是時候載入訓練(train)和驗證(validation)資料集了。

In [ ]:
DATA_LABELS = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"] 
    
class MyDataset(Dataset):
    def __init__(self, data_dir):
        self.imgs = []
        self.labels = []
        
        for l_idx, label in enumerate(DATA_LABELS):
            data_paths = glob.glob(data_dir + label + '/*.png', recursive=True)
            for path in data_paths:
                img = tv_io.read_image(path, tv_io.ImageReadMode.RGB)
                self.imgs.append(pre_trans(img).to(device))
                self.labels.append(torch.tensor(l_idx).to(device))


    def __getitem__(self, idx):
        img = self.imgs[idx]
        label = self.labels[idx]
        return img, label

    def __len__(self):
        return len(self.imgs)


選擇批次大小`n`並根據我們是在`train`訓練還是在`valid`驗證來將`shuffle`設置為`True`或`False`。參考請查看[notebook 05b](05b_presidential_doggy_door.ipynb)。

In [ ]:
n = FIXME

train_path = "data/fruits/train/"
train_data = MyDataset(train_path)
train_loader = DataLoader(train_data, batch_size=n, shuffle=FIXME)
train_N = len(train_loader.dataset)

valid_path = "data/fruits/valid/"
valid_data = MyDataset(valid_path)
valid_loader = DataLoader(valid_data, batch_size=n, shuffle=FIXME)
valid_N = len(valid_loader.dataset)


## 7.8 訓練模型


是時候訓練模型了！我們已經將`train`和`validate`函式移到了[utils.py](./utils.py)文件中。在運行以下內容之前，請確保所有變數都正確定義。

重新運行此程式碼區塊(Cell)或更改`epochs`的數量可能會有所幫助。

In [ ]:
epochs = 10

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    utils.train(my_model, train_loader, train_N, random_trans, optimizer, loss_function)
    utils.validate(my_model, valid_loader, valid_N, loss_function)


## 7.9 解凍(Unfreeze)模型進行微調


如果您已經達到了92%的驗證準確度(validation accuracy)，那麼下一步是選用的(optional)。如果沒有，我們建議使用非常低的學習率(learning rate)對模型進行微調。

In [ ]:
# Unfreeze the base model
vgg_model.requires_grad_(FIXME)
optimizer = Adam(my_model.parameters(), lr=.0001)

In [ ]:
epochs = 1

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    utils.train(my_model, train_loader, train_N, random_trans, optimizer, loss_function)
    utils.validate(my_model, valid_loader, valid_N, loss_function)

## 7.10 評估模型


希望您現在擁有一個驗證準確度為92%或更高的模型。如果沒有，您可能需要回去運行更多的訓練週期(epoch)，或調整您的資料增強(data augmentation)。

一旦您對驗證準確度感到滿意，請執行以下程式碼區塊(cell)來評估模型。評估函式將返回一個`tuple`，其中第一個值是您的損失(loss)值，第二個值是您的準確度(accuracy)。要通過，模型需要達到`92%或更高`的準確度值。

In [ ]:
utils.validate(my_model, valid_loader, valid_N, loss_function)

## 7.11 運行評量


要評估您的模型，請運行以下兩個程式碼區塊(cells)。

**注意：** `run_assessment`假設您的模型名為`my_model`。如果由於任何原因您修改了這些變數名稱，請更新傳遞給`run_assessment`的參數名稱。

In [ ]:
from run_assessment import run_assessment

In [ ]:
run_assessment(my_model)

## 7.12 產生證書(Certificate)

如果您通過了評量，請返回課程頁面（如下所示）並點擊"ASSESS TASK"按鈕，這將為您產生課程證書。

<img src="./images/assess_task.png" style="width: 800px;">


<img src="./images/assess\_task.png" style="width: 800px;">

<center><a href="https://www.nvidia.com/dli"> <img src="images/DLI_Header.png" alt="Header" style="width: 400px;"/> </a></center>


